**Методичка**
---


In [ ]:
import pandas as pd
import seaborn as sns

from sklearn import tree

df = pd.read_csv('students.csv', delimiter=',')
df_iris = pd.read_csv('iris.csv', delimiter=',')

In [ ]:
df.info()

In [ ]:
df_cut=df[['Growth','Weight','Sex','Hair length','Children number']]
df_cut=df_cut.dropna()

In [ ]:
sns.pairplot(df_cut, hue='Sex')

In [ ]:
model=tree.DecisionTreeClassifier(max_depth=2)
model.fit(df_cut[['Growth','Weight','Hair length','Children number']].values.reshape(-1,4), y=df_cut['Sex'].values)

In [ ]:
import graphviz

dot_data = tree.export_graphviz(model, out_file=None, feature_names=['Growth', 'Weight', 'Hair length', 'Children number'],
    class_names=['f', 'm'],
    filled=True, rounded=True,
    special_characters=True)

graph = graphviz.Source(dot_data)
graph

In [ ]:
df_test=pd.read_csv('students_test.csv', delimiter=',')
df_test_cut=df_test[['Growth','Weight','Sex','Hair length','Children number']]
df_test_cut=df_test_cut.dropna()

# через функцию predict прогоняем объекты тестовой выборки
df_test_cut['Predicted']=model.predict(df_test_cut[['Growth','Weight','Hair length','Children number']].values.reshape(-1,4))

In [ ]:
pd.crosstab(df_test_cut['Predicted'],df_test_cut['Sex'])

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
precision_recall_fscore_support(df_test_cut['Sex'], df_test_cut['Predicted'])

**Задание**
---


In [ ]:
df = df_iris

df.head()

In [ ]:
df_test = df.groupby('Class', group_keys=False).sample(n=15, random_state=42)
df_train = df.drop(df_test.index)

In [ ]:
df_test

In [ ]:
X_train = df_train.drop('Class', axis=1)
y_train = df_train['Class']
X_test = df_test.drop('Class', axis=1)
y_test = df_test['Class']


model = tree.DecisionTreeClassifier(random_state=42, max_depth=3)
model.fit(X_train, y_train)


dot_data = tree.export_graphviz(model, out_file=None, feature_names=['SepalLength','SepalWidth','PetalLength','PetalWidth'],
    class_names=['Iris-setoza', 'Iris-versicolor', 'Iris-virginica'],
    filled=True, rounded=True,
    special_characters=True)

graph = graphviz.Source(dot_data)
graph

In [ ]:
df_test['Predicted'] = model.predict(df_test.drop('Class', axis=1))

In [ ]:
pd.crosstab(df_test['Predicted'],df_test['Class'])

In [ ]:
# 📊 Decision Boundary с фиксированными признаками (для исходной модели)
import numpy as np
import matplotlib.pyplot as plt

f1, f2 = 'PetalLength', 'PetalWidth'
# Фиксируем остальные признаки на средних значениях из обучающей выборки
fixed_features = {
    'SepalLength': df_train['SepalLength'].mean(),
    'SepalWidth': df_train['SepalWidth'].mean()
}

# Сетка для двух признаков
x_min, x_max = df_iris[f1].min() - 1, df_iris[f1].max() + 1
y_min, y_max = df_iris[f2].min() - 1, df_iris[f2].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))

# Создаём DataFrame с 4 признаками для предсказания
grid_df = pd.DataFrame({
    f1: xx.ravel(),
    f2: yy.ravel(),
    **fixed_features  # Добавляем фиксированные признаки
})

# Предсказываем исходной моделью (важен порядок колонок!)
Z = model.predict(grid_df[['SepalLength','SepalWidth','PetalLength','PetalWidth']])
Z = Z.reshape(xx.shape)

# График
plt.figure(figsize=(10, 8))
plt.contourf(xx, yy, Z, alpha=0.4, cmap='viridis')
plt.contour(xx, yy, Z, colors='black', linewidths=0.5, alpha=0.5)

for cls in df_iris['Class'].unique():
    mask = df_iris['Class'] == cls
    plt.scatter(df_iris.loc[mask, f1], df_iris.loc[mask, f2], 
                label=cls, edgecolors='black', s=80, alpha=0.7)

plt.xlabel(f1)
plt.ylabel(f2)
plt.title('🎯 Decision Boundary (с фиксированными признаками)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
precision_recall_fscore_support(df_test_cut['Sex'], df_test_cut['Predicted'])

In [ ]:
df = pd.read_csv('students.csv', delimiter=',')

df_cut=df[['Growth','Weight','Sex']]
df_cut=df_cut.dropna()

df_cut.head()




In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(max_depth=2, random_state=0)
model.fit(df_cut[['Growth', 'Weight']].values.reshape(-1,2), y=df_cut['Sex'].values)

In [ ]:
df_test = pd.read_csv('students_test.csv', delimiter=',')

result=model.predict_proba(df_test_cut[['Growth', 'Weight']].values.reshape(-1,2))

print(result)

df_test_cut['pr class 0']=result[:,0]
df_test_cut['pr class 1']=result[:,1]
df_test_cut.head()

In [ ]:
#df_test_cut[(df_test_cut['pr class 1']<0.9) &(df_test_cut['Sex']=='мужской')].head()

In [ ]:
import numpy as np
limit = 0.6
df_test_cut['Predicted'] = np.where(df_test_cut['pr class 1'] > limit, 'мужской', 'женский')

In [ ]:
pd.crosstab(df_test_cut['Predicted'],df_test_cut['Sex'])